# Vaccination research dataset preflight

Read schema only; never print source tweets or identifiers.

In [ ]:
from pathlib import Path
import json
import pandas as pd
candidates=[
    Path("/kaggle/input/datasets/keplaxo/twitter-vaccination-dataset"),
    Path("/kaggle/input/twitter-vaccination-dataset"),
]
root=next((p for p in candidates if p.is_dir()),candidates[0])
paths=sorted(root.rglob("*.csv"))
print("Attached CSV filenames:",[p.name for p in paths])
report={"attached":root.is_dir(),"files":[]}
for path in paths:
    item={"filename":path.name,"file_size_bytes":path.stat().st_size}
    try:
        df=pd.read_csv(path,nrows=1000,dtype=str,keep_default_na=False,low_memory=False,encoding="utf-8",on_bad_lines="skip")
        item["readable"]=True
        item["columns"]=list(df.columns)
        item["sample_rows_parsed"]=len(df)
        names={x.lower().strip():x for x in df.columns}
        for group,candidates in [
            ("date_candidates",["date","created_at","datetime","timestamp","tweet_date","time","created"]),
            ("text_candidates",["text","tweet","tweet_text","full_text","content","original_text","tweet content"]),
            ("id_candidates",["tweet_id","id","id_str","status_id","tweetid"]),
            ("language_candidates",["language","lang","tweet_language","tweet_lang"])]:
            item[group]=[names[c] for c in candidates if c in names]
        # Never log original tweet texts or sample record identifiers.
        if item["date_candidates"]:
            dates=df[item["date_candidates"][0]]
            parsed=pd.to_datetime(dates,errors="coerce",utc=True,format="mixed")
            item["date_parseable_fraction"]=round(float(parsed.notna().mean()),4)
        item["has_expected_master_name"]=path.name.lower()=="master.csv"
    except Exception as exc:
        item["readable"]=False
        item["error_type"]=type(exc).__name__
    report["files"].append(item)
target=Path("/kaggle/working/schema_probe.json")
target.write_text(json.dumps(report,indent=2))
print("Schema probe finished. No tweet text was printed.")